In [1]:
from models import SEASModel
from evaluation_utils import evaluate_iplg_convergence
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
from data_utils import CSGridMLMDataset
from train_utils import full_to_partial_masking
from torch.utils.data import DataLoader
import torch
from torch.nn import CrossEntropyLoss
import os
import pickle
from train_utils import train_IPLG
from data_utils import latent_MH_collate_fn
from pprint import pprint
import pandas as pd

device_name = 'cuda:0'
batch_size = 4

/home/maximos/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
if device_name == 'cpu':
    device = torch.device('cpu')
else:
    if torch.cuda.is_available():
        device = torch.device(device_name)
    else:
        print('Selected device not available: ' + device_name)
# end device selection

In [3]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)
mask_token_id = tokenizer.mask_token_id
bar_token_id = tokenizer.bar_token_id

In [4]:
loss_scheme = 'l'

d_model = 512
transformer_model = SEASModel(
    chord_vocab_size=len(tokenizer.vocab),
    d_model=d_model,
    nhead=8,
    num_layers=8,
    grid_length=80,
    pianoroll_dim=tokenizer.pianoroll_dim,
    guidance_dim=d_model,
    device=device,
)
checkpoint = torch.load(f'saved_models/iplg/SE/iplg_{loss_scheme}_loss.pt', map_location=device_name)
transformer_model.load_state_dict(checkpoint)
transformer_model.to(device)
transformer_model.eval()

SEASModel(
  (melody_proj): Linear(in_features=13, out_features=512, bias=True)
  (harmony_embedding): Embedding(355, 512)
  (dropout): Dropout(p=0.3, inplace=False)
  (encoder_layers): ModuleList(
    (0-7): 8 x TransformerEncoderLayerWithFiLM(
      (layer): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=2048, bias=True)
        (dropout): Dropout(p=0.3, inplace=False)
        (linear2): Linear(in_features=2048, out_features=512, bias=True)
        (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.3, inplace=False)
        (dropout2): Dropout(p=0.3, inplace=False)
      )
      (film): FiLMAdapter(
        (gamma): Linear(in_features=512, out_features=512, bias=True)
        (beta): Linea

In [5]:
source_folder = 'MIDIs/jazz/real'
target_folder = 'MIDIs/nottingham/real'

source_piece_path = os.path.join(source_folder, 'real_0.mid')
target_piece_path = os.path.join(target_folder, 'real_0.mid')

In [6]:
source_encoded = tokenizer.encode(source_piece_path)
target_encoded = tokenizer.encode(target_piece_path)

In [7]:
# get the source info
source_melody_grid = torch.tensor(source_encoded['pianoroll']).unsqueeze(0).to(device)
source_harmony_ids = torch.tensor(source_encoded['harmony_ids']).unsqueeze(0).to(device)

In [8]:
# for source, we need to consider harmony fully masked, except from bar tokens
source_harmony_masks, _ = full_to_partial_masking(
                    source_harmony_ids,
                    mask_token_id,
                    0,
                    bar_token_id=bar_token_id
                )

In [9]:
# get the "natural" harmony embeddings for this melody - consider mask harmony tokens
_, source_melody_layers_output = transformer_model(source_melody_grid, source_harmony_masks, get_layers_output=True)

In [10]:
print(source_melody_layers_output[0].shape)

torch.Size([1, 1, 512])


In [11]:
target_melody_grid = torch.tensor(target_encoded['pianoroll']).unsqueeze(0).to(device)
target_harmony_ids = torch.tensor(target_encoded['harmony_ids']).unsqueeze(0).to(device)

In [12]:
# for the target harmony, we need both melody and harmony token ids inside
_, target_melody_harmony_layers_output = transformer_model(target_melody_grid, target_harmony_ids, get_layers_output=True)

In [13]:
# get the difference as activation steering
h = {}
for k,v in target_melody_harmony_layers_output.items():
    h[k] = v - source_melody_layers_output[k]

In [14]:
h_4 = {4: h[4]}
h_6 = {6: h[6]}

In [15]:
# generate 
y = transformer_model(source_melody_grid, source_harmony_masks, steering_vectors=h_4, alpha=0.5, get_layers_output=False)

In [ ]:
# get average steering from folder
target_files = os.listdir(target_folder)

target_mean_h = {}

for f in target_files:
    target_piece_path = os.path.join( target_folder, f )
    target_encoded = tokenizer.encode(target_piece_path)
    _, target_melody_harmony_layers_output = transformer_model(target_melody_grid, target_harmony_ids, get_layers_output=True)
    for k,v in target_melody_harmony_layers_output.items():
        if k in target_mean_h.keys():
            target_mean_h[k] += v/len(target_files)
        else:
            target_mean_h[k] = v/len(target_files)

TypeError: cannot unpack non-iterable int object